In [ ]:
from pathlib import Path
import json
import sys

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()),
    Path.cwd()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.audio_preprocessing import preprocess_audio_for_transcription
from src.transcription import transcribe_audio_file
from src.utils import load_environment, validate_environment

# Load and validate environment variables
load_environment()

missing_keys = validate_environment()
if missing_keys:
    raise EnvironmentError(
        f"Missing required environment variables: {', '.join(missing_keys)}"
    )

# Original podcast file
audio_path = repo_root / "sources" / "The_Blueprint_For_Trustworthy_AI.m4a"

# Preprocess audio to Whisper-compatible size
preprocessed_audio_path = preprocess_audio_for_transcription(audio_path)

print(f"Original file: {audio_path}")
print(f"Preprocessed file: {preprocessed_audio_path}")

print(f"Original size: {audio_path.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Preprocessed size: {preprocessed_audio_path.stat().st_size / (1024 * 1024):.2f} MB")

# Transcribe the compressed audio
result = transcribe_audio_file(preprocessed_audio_path)

print(json.dumps(result.metadata, indent=2, ensure_ascii=False))
print(result.text[:500])

In [1]:
from pathlib import Path
import sys

repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.pdf_processor import extract_pdf_pages

eu_ai_act_path = repo_root / "sources" / "eu_ai_act.pdf"
trustworthy_ai_path = repo_root / "sources" / "altai_final_14072020_cs_accessible2_jsd5pdf_correct-title_3AC24743-DE11-0B7C-7C891D1484944E0A_68342.pdf"

eu_pages = extract_pdf_pages(eu_ai_act_path, source_id="eu_ai_act", document_type="eu_ai_act")
trust_pages = extract_pdf_pages(trustworthy_ai_path, source_id="trustworthy_ai", document_type="trustworthy_ai")

print(f"EU AI Act pages: {len(eu_pages)}")
print(f"Trustworthy AI pages: {len(trust_pages)}")
print(eu_pages[0].metadata)
print((eu_pages[0].text or trust_pages[0].text)[:300])


EU AI Act pages: 144
Trustworthy AI pages: 34
{'filename': 'eu_ai_act.pdf', 'source_id': 'eu_ai_act', 'document_type': 'eu_ai_act', 'page_number': 1, 'total_pages': 144}
REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL
of 13 June 2024
laying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directiv es 


In [2]:
print(eu_pages[20].metadata)
print(eu_pages[20].text[:300])

{'filename': 'eu_ai_act.pdf', 'source_id': 'eu_ai_act', 'document_type': 'eu_ai_act', 'page_number': 21, 'total_pages': 144}
(72) T o address concer ns relate d to opacity and complexity of cer tain AI systems and help deplo yers to fulfil their 
oblig ations under this Regulation, transparency should be required f or high-r isk AI syste ms bef ore they are placed 
on the market or put it into ser vice. High-r isk AI syst


In [3]:
print(trust_pages[10].metadata)
print(trust_pages[10].text[:300])

{'filename': 'altai_final_14072020_cs_accessible2_jsd5pdf_correct-title_3AC24743-DE11-0B7C-7C891D1484944E0A_68342.pdf', 'source_id': 'trustworthy_ai', 'document_type': 'trustworthy_ai', 'page_number': 11, 'total_pages': 34}
10 
General Safety 
• Did you define risks, risk metrics and risk levels of the AI system in each specific use 
case? 
o Did you put in place a process to continuously measure and assess risks? 
o Did you inform end-users and subjects of existing or potential risks? 
• Did you identify the possible 


In [ ]:
from src.normalization import normalize_pdf_page, normalize_transcription

normalized_pdf = normalize_pdf_page(eu_pages[0])
normalized_transcript = normalize_transcription(result)

print(normalized_pdf)
print(normalized_transcript)
print(type(normalized_pdf).__name__, type(normalized_transcript).__name__)
print([field for field in normalized_pdf.__dataclass_fields__.keys()])
print([field for field in normalized_transcript.__dataclass_fields__.keys()])
print(normalized_pdf.document_id)
print(normalized_transcript.document_id)
